# Agent的高级用法-ToolStrategy
## 1.ToolStrategy的多种结构化输出方式：schema参数
### 1.1 Pydantic类型
使用ds模型

In [ ]:
import os

from langchain.agents import create_agent
from langchain_core.messages import HumanMessage
from langchain_deepseek import ChatDeepSeek
from dotenv import load_dotenv
from scripts.regsetup import description
from rich import print as rprint
#1.读取.env配置文件信息,相关的环境变量以.env文件中的优先
load_dotenv(verbose=True)
DEEPSEEK_API_KEY=os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL=os.getenv("DEEPSEEK_BASE_URL")
#2.模型初始化
model=ChatDeepSeek(
    model="deepseek-v4-flash",
    api_key=DEEPSEEK_API_KEY,
    api_base=DEEPSEEK_BASE_URL,
    # 关键修改：关闭思考模式
    extra_body={
        "thinking": {
            "type": "disabled"
        }
    },
)

举例1：

In [ ]:

from langchain.messages import HumanMessage
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from pydantic import BaseModel, Field
from rich import print as rprint
#使用Pydantic结构化方式定义
class ContractInfo(BaseModel):
    """用户的联系方式"""
    name:str=Field(description="用户的姓名")
    email:str=Field(description="用户的邮箱")
    phone:str=Field(description="用户的电话")
agent=create_agent(
    model=model,
    response_format=ToolStrategy(schema=ContractInfo)
)
response=agent.invoke({
    "messages":[
        HumanMessage(content="从这段话中抽取结构化信息：小明的邮箱是1465465@163.com,电话是11654654651")
    ]
})
rprint(response)

举例2：添加工具的调用

In [ ]:
from langchain_core.messages import SystemMessage
from pydantic import BaseModel, Field
from typing import Literal, TypedDict, Annotated
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain.tools import tool
from rich import print as rprint

# 定义工具
@tool(parse_docstring=True)
def search_customer_database(query: str) -> str:
    """
    在客户数据库中搜索信息

    Args:
    query (str): 客户查询字符串，例如 "张三" 或 "李四"

    Returns:
    str: 客户记录字符串，包含客户姓名、等级、最近购买日期和累计消费
    """
    # 模拟数据库查询结果
    if "张三" in query.lower():
        return "客户记录：张三，VIP客户，最近购买日期：2026-01-15，累计消费：$15,000"
    elif "李四" in query.lower():
        return "客户记录：李四，普通客户，最近购买日期：2025-12-20，累计消费：$3,200"
    else:
        return f"关于客户{query}，无记录"


@tool(parse_docstring=True)
def send_email(customer: str) -> str:
    """
    发送感谢邮件

    Args:
    customer (str): 客户名称，例如 "张三" 或 "李四"

    Returns:
    str: 确认消息，包含已发送的客户名称"""
    return f"已向 {customer} 发送感谢邮件"


# 定义Pydantic Schema
class CustomerAnalysis(BaseModel):
    """
    客户分析报告
    """
    customer_name: str = Field(None, description="客户姓名")
    customer_tier: Literal["潜在客户", "普通客户", "VIP客户", "流失风险"] =Field("潜在客户",description="客户等级,只能是潜在客户、普通客户、VIP客户或流失风险")
    recent_activity: str = Field(None, description="最近活动")
    spending_level: Literal["低", "中", "高"] = Field(None, description="消费水平")
    send_email: bool = Field(False, description="是否已发送感谢邮件")


# 创建智能体
agent = create_agent(
    model=model,
    system_prompt=SystemMessage(content=""
    "请分析指定客户的情况："
    "1. 先搜索客户数据库了解最新情况 "
    "2. 如果是VIP客户，则发送感谢邮件 "
    "3. 基于搜索结果生成结构化分析报告 "
    "4. 如果用户提问与客户记录无关或找不到客户信息，则返回空对象，不发送感谢邮件"
    ),
    tools=[search_customer_database, send_email],
    response_format=ToolStrategy(CustomerAnalysis)
)

# 执行分析
result = agent.invoke({
    "messages": [{"role": "user", "content": "请分析客户张三"}]
    # "messages": [{"role": "user","content": "请分析客户李四"}]
    # "messages": [{"role": "user","content": "请分析客户王五"}]
    # "messages": [{"role": "user","content": "今天天气如何"}]
})

# 处理结果
rprint(result)
# if "structured_response" in result:
#     analysis = result["structured_response"]
#     print(analysis)

### 1.2TypeDict类型
举例1：

In [ ]:
from typing import TypedDict,Annotated
from langchain.messages import HumanMessage
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from pydantic import BaseModel, Field
from rich import print as rprint
#使用Pydantic结构化方式定义
class ContactInfo(TypedDict):
    """用户的联系方式"""
    name:Annotated[str,...,"用户姓名"]
    email:Annotated[str,...,"用户邮箱"]
    phone:Annotated[str,...,"用户的手机号"]
agent=create_agent(
    model=model,
    response_format=ToolStrategy(schema=ContactInfo)
)
response=agent.invoke({
    "messages":[
        HumanMessage(content="从这段话中抽取结构化信息：小明的邮箱是1465465@163.com,电话是11654654651")
    ]
})
rprint(response)

举例2：

In [ ]:
from langchain_core.messages import SystemMessage
from pydantic import BaseModel, Field
from typing import Literal, TypedDict, Annotated, Optional
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain.tools import tool
from rich import print as rprint

# 定义工具
@tool(parse_docstring=True)
def search_customer_database(query: str) -> str:
    """
    在客户数据库中搜索信息

    Args:
    query (str): 客户查询字符串，例如 "张三" 或 "李四"

    Returns:
    str: 客户记录字符串，包含客户姓名、等级、最近购买日期和累计消费
    """
    # 模拟数据库查询结果
    if "张三" in query.lower():
        return "客户记录：张三，VIP客户，最近购买日期：2026-01-15，累计消费：$15,000"
    elif "李四" in query.lower():
        return "客户记录：李四，普通客户，最近购买日期：2025-12-20，累计消费：$3,200"
    else:
        return f"关于客户{query}，无记录"


@tool(parse_docstring=True)
def send_email(customer: str) -> str:
    """
    发送感谢邮件

    Args:
    customer (str): 客户名称，例如 "张三" 或 "李四"

    Returns:
    str: 确认消息，包含已发送的客户名称"""
    return f"已向 {customer} 发送感谢邮件"


# 使用 TypedDict 定义客户分析报告 Schema
class CustomerAnalysis(TypedDict):
    """客户分析报告"""
    customer_name: Annotated[Optional[str], None, "客户姓名"]
    customer_tier: Annotated[Literal["潜在客户", "普通客户", "VIP客户", "流失风险"], "潜在客户", "客户等级"]
    recent_activity: Annotated[Optional[str], None, "最近活动"]
    spending_level: Annotated[Optional[Literal["低", "中", "高"]], None, "消费水平"]
    send_email: Annotated[bool, False, "是否已发送感谢邮件"]



# 创建智能体
agent = create_agent(
    model=model,
    system_prompt=SystemMessage(content=""
    "请分析指定客户的情况："
    "1. 先搜索客户数据库了解最新情况 "
    "2. 如果是VIP客户，则发送感谢邮件 "
    "3. 基于搜索结果生成结构化分析报告 "
    "4. 如果用户提问与客户记录无关或找不到客户信息，则返回空对象，不发送感谢邮件"
    ),
    tools=[search_customer_database, send_email],
    response_format=ToolStrategy(CustomerAnalysis)
)

# 执行分析
result = agent.invoke({
    "messages": [{"role": "user", "content": "请分析客户张三"}]
    # "messages": [{"role": "user","content": "请分析客户李四"}]
    # "messages": [{"role": "user","content": "请分析客户王五"}]
    # "messages": [{"role": "user","content": "今天天气如何"}]
})

# 处理结果
rprint(result)
# if "structured_response" in result:
#     analysis = result["structured_response"]
#     print(analysis)

### 1.3JsonSchema类型
举例1：

In [ ]:
from typing import TypedDict,Annotated
from langchain.messages import HumanMessage
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from pydantic import BaseModel, Field
from rich import print as rprint

json_schema = {
    "title": "ContactInfo",
    "description": "用户的联系方式",
    "type": "object",
    "properties": {
        "name": {
            "description": "用户姓名",
            "type": "string"
        },
        "email": {
            "description": "用户邮箱地址",
            "type": "string"
        },
        "phone": {
            "description": "用户的手机号",
            "type": "string"
        }
    },
    "required": [
        "name",
        "email",
        "phone"
    ]
}
agent=create_agent(
    model=model,
    response_format=ToolStrategy(schema=json_schema)
)
response=agent.invoke({
    "messages":[
        HumanMessage(content="从这段话中抽取结构化信息：小明的邮箱是1465465@163.com,电话是11654654651")
    ]
})
rprint(response)

举例2：

In [ ]:
from langchain_core.messages import SystemMessage
from pydantic import BaseModel, Field
from typing import Literal, TypedDict, Annotated, Optional
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain.tools import tool
from rich import print as rprint

# 定义工具
@tool(parse_docstring=True)
def search_customer_database(query: str) -> str:
    """
    在客户数据库中搜索信息

    Args:
    query (str): 客户查询字符串，例如 "张三" 或 "李四"

    Returns:
    str: 客户记录字符串，包含客户姓名、等级、最近购买日期和累计消费
    """
    # 模拟数据库查询结果
    if "张三" in query.lower():
        return "客户记录：张三，VIP客户，最近购买日期：2026-01-15，累计消费：$15,000"
    elif "李四" in query.lower():
        return "客户记录：李四，普通客户，最近购买日期：2025-12-20，累计消费：$3,200"
    else:
        return f"关于客户{query}，无记录"


@tool(parse_docstring=True)
def send_email(customer: str) -> str:
    """
    发送感谢邮件

    Args:
    customer (str): 客户名称，例如 "张三" 或 "李四"

    Returns:
    str: 确认消息，包含已发送的客户名称"""
    return f"已向 {customer} 发送感谢邮件"


# 使用 json_schema 定义客户分析报告 Schema

customer_analysis_schema = {
    "title": "CustomerAnalysis",
    "type": "object",
    "description": "客户分析报告",
    "properties": {
        "customer_name": {
            "type": "string",
            "default": "",
            "description": "客户姓名"
        },
        "customer_tier": {
            "type": "string",
            "enum": ["潜在客户", "普通客户", "VIP客户", "流失风险"],
            "default": "潜在客户",
            "description": "客户等级"
        },
        "recent_activity": {
            "type": "string",
            "default": "",
            "description": "最近活动"
        },
        "spending_level": {
            "type": "string",
            "enum": ["低", "中", "高"],
            "default": "低",
            "description": "消费水平"
        },
        "send_email": {
            "type": "boolean",
            "default": False,
            "description": "是否已发送感谢邮件"
        }
    },
    # 所有字段都是必须输出的
    "required": ["customer_name", "customer_tier", "recent_activity",
                 "spending_level"]
}


# 创建智能体
agent = create_agent(
    model=model,
    system_prompt=SystemMessage(content=""
    "请分析指定客户的情况："
    "1. 先搜索客户数据库了解最新情况 "
    "2. 如果是VIP客户，则发送感谢邮件 "
    "3. 基于搜索结果生成结构化分析报告 "
    "4. 如果用户提问与客户记录无关或找不到客户信息，则返回空对象，不发送感谢邮件"
    ),
    tools=[search_customer_database, send_email],
    response_format=ToolStrategy(customer_analysis_schema)
)

# 执行分析
result = agent.invoke({
    "messages": [{"role": "user", "content": "请分析客户张三"}]
    # "messages": [{"role": "user","content": "请分析客户李四"}]
    # "messages": [{"role": "user","content": "请分析客户王五"}]
    # "messages": [{"role": "user","content": "今天天气如何"}]
})

# 处理结果
rprint(result)
# if "structured_response" in result:
#     analysis = result["structured_response"]
#     print(analysis)

### 1.4@dataclass类型
举例1：

In [ ]:

from langchain.messages import HumanMessage
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from pydantic import BaseModel, Field
from rich import print as rprint
from dataclasses import dataclass
#使用Pydantic结构化方式定义
@dataclass
class ContractInfo:
    """用户的联系方式"""
    name:str=Field(description="用户的姓名")
    email:str=Field(description="用户的邮箱")
    phone:str=Field(description="用户的电话")
agent=create_agent(
    model=model,
    response_format=ToolStrategy(schema=ContractInfo)
)
response=agent.invoke({
    "messages":[
        HumanMessage(content="从这段话中抽取结构化信息：小明的邮箱是1465465@163.com,电话是11654654651")
    ]
})
rprint(response)

举例2：

In [ ]:
from langchain_core.messages import SystemMessage
from pydantic import BaseModel, Field
from typing import Literal, TypedDict, Annotated
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain.tools import tool
from rich import print as rprint
from dataclasses import dataclass
# 定义工具
@tool(parse_docstring=True)
def search_customer_database(query: str) -> str:
    """
    在客户数据库中搜索信息

    Args:
    query (str): 客户查询字符串，例如 "张三" 或 "李四"

    Returns:
    str: 客户记录字符串，包含客户姓名、等级、最近购买日期和累计消费
    """
    # 模拟数据库查询结果
    if "张三" in query.lower():
        return "客户记录：张三，VIP客户，最近购买日期：2026-01-15，累计消费：$15,000"
    elif "李四" in query.lower():
        return "客户记录：李四，普通客户，最近购买日期：2025-12-20，累计消费：$3,200"
    else:
        return f"关于客户{query}，无记录"


@tool(parse_docstring=True)
def send_email(customer: str) -> str:
    """
    发送感谢邮件

    Args:
    customer (str): 客户名称，例如 "张三" 或 "李四"

    Returns:
    str: 确认消息，包含已发送的客户名称"""
    return f"已向 {customer} 发送感谢邮件"

@dataclass
class CustomerAnalysis:
    """
    客户分析报告
    """
    customer_name: str = Field(None, description="客户姓名")
    customer_tier: Literal["潜在客户", "普通客户", "VIP客户", "流失风险"] =Field("潜在客户",description="客户等级,只能是潜在客户、普通客户、VIP客户或流失风险")
    recent_activity: str = Field(None, description="最近活动")
    spending_level: Literal["低", "中", "高"] = Field(None, description="消费水平")
    send_email: bool = Field(False, description="是否已发送感谢邮件")


# 创建智能体
agent = create_agent(
    model=model,
    system_prompt=SystemMessage(content=""
    "请分析指定客户的情况："
    "1. 先搜索客户数据库了解最新情况 "
    "2. 如果是VIP客户，则发送感谢邮件 "
    "3. 基于搜索结果生成结构化分析报告 "
    "4. 如果用户提问与客户记录无关或找不到客户信息，则返回空对象，不发送感谢邮件"
    ),
    tools=[search_customer_database, send_email],
    response_format=ToolStrategy(CustomerAnalysis)
)

# 执行分析
result = agent.invoke({
    "messages": [{"role": "user", "content": "请分析客户张三"}]
    # "messages": [{"role": "user","content": "请分析客户李四"}]
    # "messages": [{"role": "user","content": "请分析客户王五"}]
    # "messages": [{"role": "user","content": "今天天气如何"}]
})

# 处理结果
rprint(result)
# if "structured_response" in result:
#     analysis = result["structured_response"]
#     print(analysis)

### 1.5多schema联合模式
举例1：

In [ ]:
from pydantic import BaseModel, Field
from typing import Union
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain.messages import HumanMessage


class ContactInfo(BaseModel):
    """用户的联系方式"""
    name: str = Field(description="用户姓名")
    email: str = Field(description="用户邮箱地址")
    phone: str = Field(description="用户的手机号")


class EventInfo(BaseModel):
    """事件详情"""
    event_name: str = Field(description="事件名称")
    date: str = Field(description="事件发生日期")


agent = create_agent(
    model=model,
    response_format=ToolStrategy(
        schema=Union[ContactInfo, EventInfo]
    )
)

response = agent.invoke(
    {
        "messages": [
            HumanMessage("从这段话中抽取结构化信息：小明的邮箱地址为：shkstart@atguigu.com，手机号：12345678912")
        ]
    }
)

for msg in response["messages"]:
    msg.pretty_print()
print(response["structured_response"])